In [ ]:
# https://musicbrainz.org/doc/MusicBrainz_API

In [1]:
# Uncomment if needed
%pip install requests pandas python-dotenv rich

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.9-cp312-cp312-macosx_10_13_universal2.whl.metadata (41 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.9-cp312-cp312-macosx_10_13_universal2.whl (319 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 9.3 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 12.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [rich]2m10/11 [rich]own-it-py]zer]
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pprint import pprint

import pandas as pd
import requests

BASE_URL = "https://musicbrainz.org/ws/2"

HEADERS = {
    # MusicBrainz requests that applications identify themselves.
    "User-Agent": "SessionResearchNotebook/0.1 (noodnik2@gmail.com)"
}

In [4]:
def mb_get(endpoint: str, **params):
    params["fmt"] = "json"

    url = f"{BASE_URL}/{endpoint}"

    r = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30,
    )

    r.raise_for_status()
    return r.json()

In [5]:
artist_name = "Nathan Ross"

result = mb_get(
    "artist",
    query=f'artist:"{artist_name}"',
    limit=5,
)

pprint(result["artists"])

[{'aliases': [{'begin-date': None,
               'end-date': None,
               'locale': None,
               'name': 'Nat Ross',
               'primary': None,
               'sort-name': 'Ross, Nat',
               'type': None}],
  'id': 'ef1a8f95-b0a8-4827-b725-4947f137cc20',
  'life-span': {'ended': None},
  'name': 'Nathan Ross',
  'score': 100,
  'sort-name': 'Ross, Nathan',
  'type': 'Person',
  'type-id': 'b6e035f4-3ce9-331c-97df-83397230b0df'}]


In [6]:
# TODO retrieve this from the result above
artist_id = "ef1a8f95-b0a8-4827-b725-4947f137cc20"

artist = mb_get(
    f"artist/{artist_id}",
    inc="artist-rels+recording-rels+release-rels+url-rels"
)

print(artist.keys())

dict_keys(['relations', 'begin-area', 'id', 'life-span', 'end-area', 'type-id', 'disambiguation', 'gender-id', 'name', 'type', 'isnis', 'sort-name', 'country', 'ipis', 'area', 'gender'])


In [7]:
pprint(artist["relations"][0], sort_dicts=False)

{'direction': 'forward',
 'begin': '1953-01-20',
 'attribute-ids': {'violin': '089f123c-0f7d-4105-a64e-49de81ca8fa4'},
 'target-type': 'recording',
 'attribute-credits': {},
 'target-credit': '',
 'end': '1953-01-20',
 'source-credit': '',
 'type-id': '59054b12-01ac-43ee-a618-285fd397e461',
 'recording': {'id': 'fb0fd734-a880-4984-9dfb-011f7a7188fb',
               'title': 'Blue Gardenia',
               'disambiguation': '1953-01-20',
               'length': 181480,
               'video': False},
 'ended': True,
 'attributes': ['violin'],
 'type': 'instrument',
 'attribute-values': {}}


In [8]:
rows = []

for rel in artist.get("relations", []):
    rows.append({
        "type": rel.get("type"),
        "direction": rel.get("direction"),
        "begin": rel.get("begin"),
        "end": rel.get("end"),
        "attributes": rel.get("attributes"),
        "target_type": rel.get("target-type"),
    })

df = pd.DataFrame(rows)
df.head()

,type,direction,begin,end,attributes,target_type
0,instrument,forward,1953-01-20,1953-01-20,[violin],recording
1,instrument,forward,1954-07-27,1954-07-27,[violin],recording
2,instrument,forward,1954-12-20,1954-12-20,[violin],recording
3,instrument,forward,1954-12-20,1954-12-20,[violin],recording
4,instrument,forward,1954-12-20,1954-12-20,[violin],recording
